In [0]:
-- SUB QUERY AND AGGREGATION FUNCTIONS
SELECT
  *
FROM
  canada_sales.v01.orders;

SELECT
  customerid,
  productid,
  SUM(quantity) AS Total_quantity,
  SUM(orderamt) AS Total_amt
FROM
  canada_sales.v01.orders
WHERE
  orderamt
    > (
      SELECT
        AVG(orderamt)
      FROM
        canada_sales.v01.orders
    )
GROUP BY
  ALL
ORDER BY customerid, productid;

-- SQL JOINs
-- INNER JOINS
SELECT
  *
FROM
  canada_sales.v01.customers;

SELECT
  *
FROM
  canada_sales.v01.orders;

--# Common column is customerid
--- INNER JOIN on customerid
SELECT
  c.customername,
  SUM(o.orderamt) AS total_amt
FROM
  canada_sales.v01.customers c
    INNER JOIN canada_sales.v01.orders o
      ON c.customerid = o.customerid
GROUP BY
  c.customername;

-- # LEFT JOIN
SELECT
  *
FROM
  (
    SELECT
      c.customername,
      o.orderamt
    FROM
      canada_sales.v01.customers c
        LEFT JOIN canada_sales.v01.orders o
          ON c.customerid = o.customerid
  )
WHERE
  orderamt > 20000
  AND customername LIKE 'Nova%';

--- CASE STATEMENT
SELECT
  *,
  CASE
    WHEN total_amt > 5000000 THEN 'HIGH'
    WHEN total_amt > 1000000 THEN 'MEDIUM'
    ELSE 'LOW'
  END AS customer_segment
FROM
  (
    SELECT
      c.customername,
      SUM(o.orderamt) AS total_amt
    FROM
      canada_sales.v01.customers c
        INNER JOIN canada_sales.v01.orders o
          ON c.customerid = o.customerid
    GROUP BY
      c.customername
  )
ORDER BY
  total_amt DESC;

--- ROLLUP AND CUBE
SELECT
  c.customername,
  o.orderdate,
  SUM(o.orderamt) AS total_amt
FROM
  canada_sales.v01.customers c
    INNER JOIN canada_sales.v01.orders o
      ON c.customerid = o.customerid
WHERE
  c.customername LIKE 'Ac%'
GROUP BY
  ROLLUP(c.customername, o.orderdate)
ORDER BY
  c.customername NULLS LAST,
  o.orderdate NULLS LAST;

SELECT
  c.customername,
  o.orderdate,
  SUM(o.orderamt) AS total_amt
FROM
  canada_sales.v01.customers c
    INNER JOIN canada_sales.v01.orders o
      ON c.customerid = o.customerid
WHERE
  c.customername LIKE 'Ac%'
GROUP BY
  CUBE(c.customername, o.orderdate)
ORDER BY
  c.customername NULLS LAST,
  o.orderdate NULLS LAST;

--- ROW_NUMBER() OVER (PARTITION BY a column) AS RowNum
SELECT
  orderdate,
  customerid,
  productid,
  orderamt,
  ROW_NUMBER() OVER (PARTITION BY customerid, productid ORDER BY orderdate) AS RowNum
FROM
  canada_sales.v01.orders;


  --- SUM() OVER()...to get running total
SELECT
  orderdate,
  customerid,
  productid,
  orderamt,
  SUM(orderamt) OVER (PARTITION BY customerid ORDER BY orderdate, customerid, productid) AS RunningTotal
FROM
  canada_sales.v01.orders;

SELECT
    city, COUNT(*)
FROM canada_sales.v01.customers
WHERE city LIKE 'T%'
GROUP BY city
HAVING COUNT(*)>1;